In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
from tkinter import ttk
import re
from PIL import Image, ImageTk
import random
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Dummy credentials for login
USERNAME = "admin"
PASSWORD = "password"

# Variables to store uploaded data
saved_text_content = None
personal_details = {}
cv_content = None
job_recommendations = ["Job A - New York \n Link: https...", "Job B - Los Angeles \n Link: https...", "Job C - Chicago \n Link: https...", "Job D - Houston \n Link: https...", "Job E - Phoenix \n Link: https...", "Job F - Philadelphia \n Link: https..."]

# Email validation function
def is_valid_email(email):
    return re.match(r"[^@]+@[^@]+\.[^@]+", email)

# Phone number validation function
def is_valid_phone(phone):
    return re.match(r"^\+?[0-9]{10,15}$", phone)

# Function to handle login
def login():
    username = username_entry.get()
    password = password_entry.get()
    if username == USERNAME and password == PASSWORD:
        login_window.destroy()
        show_dashboard(username)
    else:
        messagebox.showerror("Login Failed", "Invalid username or password")

# Function to show the main window
def show_dashboard(username):
    global profile_tab, root, recommended_jobs_frame, start_job_hunting_button, no_jobs_label
    root = tk.Tk()
    root.title("Dashboard")
    root.configure(bg="#1e1e1e")

    # Set the dimensions to resemble an iPhone screen
    root.geometry("375x667")

    # Create a frame for better organization
    frame = ttk.Frame(root, padding=20)
    frame.place(relx=0.1, rely=0.1, relwidth=0.8, relheight=0.8)

    # Create a Label for the Text widget
    welcome_label = ttk.Label(frame, text=f"Welcome to JobHunterAI, {username}:")
    welcome_label.pack(pady=10)

    # Create a Notebook (tabbed interface)
    notebook = ttk.Notebook(frame)
    notebook.pack(expand=True, fill='both')

    # Create the 'Main' tab
    main_tab = ttk.Frame(notebook)
    notebook.add(main_tab, text="Main")

    # Create a section for recommended jobs
    recommended_jobs_frame = ttk.Frame(main_tab)
    recommended_jobs_frame.pack(pady=10, fill=tk.BOTH, expand=True)
    
    # Placeholder for recommended jobs
    if 'cv_uploaded' not in personal_details:
        no_jobs_label = ttk.Label(recommended_jobs_frame, text="No new recommended jobs available.")
        no_jobs_label.pack(pady=5)
        
        start_job_hunting_button = ttk.Button(recommended_jobs_frame, text="Start Job Hunting", command=check_personal_details)
        start_job_hunting_button.pack(pady=5)
    else:
        display_job_recommendations()

    # Create the 'Profile Settings' tab
    profile_tab = ttk.Frame(notebook)
    notebook.add(profile_tab, text="Profile Settings")

    # Add content to the 'Profile Settings' tab
    profile_label = ttk.Label(profile_tab, text="Profile Settings")
    profile_label.pack(pady=10)

    # Display saved personal details in the profile tab
    display_profile_details(profile_tab)

    # Start the GUI event loop
    root.mainloop()

def display_profile_details(tab):
    for widget in tab.winfo_children():
        widget.destroy()
    
    if personal_details:
        for key, value in personal_details.items():
            if key != 'password' and key != 'cv_uploaded':
                detail_label = ttk.Label(tab, text=f"{key.capitalize()}: {value}")
                detail_label.pack(anchor='w', pady=2)
    else:
        no_details_label = ttk.Label(tab, text="No personal details available.")
        no_details_label.pack(anchor='w', pady=2)
        
    edit_details_button = ttk.Button(tab, text="Edit Details", command=edit_personal_details)
    edit_details_button.pack(pady=5)

# Function to check if personal details are filled in before starting job hunting
def check_personal_details():
    if not personal_details:
        messagebox.showinfo("Information", "Please fill in your personal details in the 'Profile Settings' tab before starting job hunting.")
    else:
        start_job_hunting()

# Function to start job hunting
def start_job_hunting():
    if 'job_hunting_window' in globals():
        job_hunting_window.destroy()
    job_hunting_window = tk.Toplevel()
    job_hunting_window.title("Start Job Hunting")
    job_hunting_window.geometry("375x667")
    ask_for_cv(job_hunting_window)

def ask_for_cv(window):
    for widget in window.winfo_children():
        widget.destroy()
    
    frame = ttk.Frame(window, padding=20)
    frame.pack(expand=True, fill=tk.BOTH)

    cv_label = ttk.Label(frame, text="Upload your CV:")
    cv_label.pack(anchor='w')
    
    upload_cv_button = ttk.Button(frame, text="Upload CV", command=lambda: upload_cv(window))
    upload_cv_button.pack(pady=10)
    
    if 'phone' not in personal_details:
        phone_label = ttk.Label(frame, text="Phone (optional for updates):")
        phone_label.pack(anchor='w')
        phone_entry = ttk.Entry(frame)
        phone_entry.pack(pady=5)
    else:
        phone_entry = None
    
    email_updates_var = tk.BooleanVar()
    email_updates_check = ttk.Checkbutton(frame, text="Send updates via email", variable=email_updates_var)
    email_updates_check.pack(pady=5)
    
    phone_updates_var = tk.BooleanVar()
    phone_updates_check = ttk.Checkbutton(frame, text="Send updates via phone number", variable=phone_updates_var)
    phone_updates_check.pack(pady=5)

    save_details_button = ttk.Button(frame, text="Submit", command=lambda: save_job_hunting_details(phone_entry, email_updates_var.get(), phone_updates_var.get(), window))
    save_details_button.pack(pady=10)

    cancel_button = ttk.Button(frame, text="Cancel", command=window.destroy)
    cancel_button.pack(pady=5)

def upload_cv(window):
    global cv_content
    file_path = filedialog.askopenfilename(filetypes=[("PDF files", "*.pdf")])
    if file_path:
        with open(file_path, "rb") as file:
            cv_content = file.read()
        personal_details['cv_uploaded'] = True
        messagebox.showinfo("Success", "CV uploaded successfully!")

def save_job_hunting_details(phone_entry, email_updates, phone_updates, window):
    if phone_entry:
        phone = phone_entry.get()
        if phone and not is_valid_phone(phone):
            messagebox.showerror("Invalid Phone Number", "Please enter a valid phone number.")
            return
        personal_details['phone'] = phone
    personal_details['email_updates'] = email_updates
    personal_details['phone_updates'] = phone_updates
    display_job_recommendations(window)

def display_job_recommendations(window=None):
    if window:
        window.destroy()
    
    if recommended_jobs_frame.winfo_children():
        for widget in recommended_jobs_frame.winfo_children():
            widget.destroy()
    
    recommendations = random.sample(job_recommendations, 3)
    for job in recommendations:
        job_label = ttk.Label(recommended_jobs_frame, text=job)
        job_label.pack(pady=2)

    send_to_mail_button = ttk.Button(recommended_jobs_frame, text="Send to Mail", command=send_to_mail)
    send_to_mail_button.pack(pady=10)

    start_new_job_hunting_button = ttk.Button(recommended_jobs_frame, text="Start New Job Hunting", command=check_personal_details)
    start_new_job_hunting_button.pack(pady=10)

def send_to_mail():
    recommendations = random.sample(job_recommendations, 3)
    email_content = "<h1>Job Recommendations</h1><ul>"
    for job in recommendations:
        email_content += f"<li>{job}</li>"
    email_content += "</ul>"
    send_email(personal_details['email'], "Job Recommendations", email_content)

def send_email(to_email, subject, body):
    from_email = "johhunterugent@gmail.com"
    from_password = "MikailMalcikan"

    message = MIMEMultipart()
    message["From"] = from_email
    message["To"] = to_email
    message["Subject"] = subject
    message.attach(MIMEText(body, "html"))

    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(from_email, from_password)
        text = message.as_string()
        server.sendmail(from_email, to_email, text)
        server.quit()
        messagebox.showinfo("Success", "Email sent successfully!")
    except Exception as e:
        messagebox.showerror("Error", f"Failed to send email: {e}")

def edit_personal_details():
    if 'details_window' in globals():
        details_window.destroy()
    details_window = tk.Toplevel()
    details_window.title("Edit Personal Details")
    details_window.geometry("375x667")
    
    frame = ttk.Frame(details_window, padding=20)
    frame.pack(expand=True, fill=tk.BOTH)
    
    details_label = ttk.Label(frame, text="Enter your personal details:")
    details_label.pack(pady=10)
    
    first_name_label = ttk.Label(frame, text="First Name:")
    first_name_label.pack(anchor='w')
    first_name_entry = ttk.Entry(frame)
    first_name_entry.insert(0, personal_details.get('first_name', ''))
    first_name_entry.pack(pady=5)
    
    last_name_label = ttk.Label(frame, text="Last Name:")
    last_name_label.pack(anchor='w')
    last_name_entry = ttk.Entry(frame)
    last_name_entry.insert(0, personal_details.get('last_name', ''))
    last_name_entry.pack(pady=5)
    
    email_label = ttk.Label(frame, text="Email:")
    email_label.pack(anchor='w')
    email_entry = ttk.Entry(frame)
    email_entry.insert(0, personal_details.get('email', ''))
    email_entry.pack(pady=5)
    
    phone_label = ttk.Label(frame, text="Phone (optional):")
    phone_label.pack(anchor='w')
    phone_entry = ttk.Entry(frame)
    phone_entry.insert(0, personal_details.get('phone', ''))
    phone_entry.pack(pady=5)
    
    address_label = ttk.Label(frame, text="Address:")
    address_label.pack(anchor='w')
    address_entry = ttk.Entry(frame)
    address_entry.insert(0, personal_details.get('address', ''))
    address_entry.pack(pady=5)
    
    postcode_label = ttk.Label(frame, text="Postcode:")
    postcode_label.pack(anchor='w')
    postcode_entry = ttk.Entry(frame)
    postcode_entry.insert(0, personal_details.get('postcode', ''))
    postcode_entry.pack(pady=5)
    
    password_label = ttk.Label(frame, text="Password:")
    password_label.pack(anchor='w')
    password_entry = ttk.Entry(frame, show="*")
    password_entry.pack(pady=5)
    
    confirm_password_label = ttk.Label(frame, text="Confirm Password:")
    confirm_password_label.pack(anchor='w')
    confirm_password_entry = ttk.Entry(frame, show="*")
    confirm_password_entry.pack(pady=5)
    
    save_details_button = ttk.Button(frame, text="Save Details", command=lambda: save_details(first_name_entry.get(), last_name_entry.get(), email_entry.get(), phone_entry.get(), address_entry.get(), postcode_entry.get(), password_entry.get(), confirm_password_entry.get(), details_window))
    save_details_button.pack(pady=10)

    cancel_button = ttk.Button(frame, text="Cancel", command=details_window.destroy)
    cancel_button.pack(pady=5)

def save_details(first_name, last_name, email, phone, address, postcode, password, confirm_password, window):
    global personal_details
    if not is_valid_email(email):
        messagebox.showerror("Invalid Email", "Please enter a valid email address.")
        return
    if phone and not is_valid_phone(phone):
        messagebox.showerror("Invalid Phone Number", "Please enter a valid phone number.")
        return
    if password and password != confirm_password:
        messagebox.showerror("Error", "Passwords do not match.")
        return
    personal_details['first_name'] = first_name
    personal_details['last_name'] = last_name
    personal_details['email'] = email
    personal_details['phone'] = phone
    personal_details['address'] = address
    personal_details['postcode'] = postcode
    if password:
        personal_details['password'] = password
    messagebox.showinfo("Success", "Personal details saved successfully!")
    window.destroy()
    display_profile_details(profile_tab)
    if 'cv_uploaded' not in personal_details:
        ask_for_cv(tk.Toplevel())

# Function to handle registration
def register():
    if 'register_window' in globals():
        register_window.destroy()
    register_window = tk.Toplevel()
    register_window.title("Register")
    register_window.geometry("375x667")
    
    frame = ttk.Frame(register_window, padding=20)
    frame.pack(expand=True, fill=tk.BOTH)
    
    # Create form fields for registration
    details_label = ttk.Label(frame, text="Enter your registration details:")
    details_label.pack(pady=10)
    
    first_name_label = ttk.Label(frame, text="First Name:")
    first_name_label.pack(anchor='w')
    first_name_entry = ttk.Entry(frame)
    first_name_entry.pack(pady=5)
    
    last_name_label = ttk.Label(frame, text="Last Name:")
    last_name_label.pack(anchor='w')
    last_name_entry = ttk.Entry(frame)
    last_name_entry.pack(pady=5)
    
    email_label = ttk.Label(frame, text="Email:")
    email_label.pack(anchor='w')
    email_entry = ttk.Entry(frame)
    email_entry.pack(pady=5)
    
    phone_label = ttk.Label(frame, text="Phone (optional):")
    phone_label.pack(anchor='w')
    phone_entry = ttk.Entry(frame)
    phone_entry.pack(pady=5)
    
    address_label = ttk.Label(frame, text="Address:")
    address_label.pack(anchor='w')
    address_entry = ttk.Entry(frame)
    address_entry.pack(pady=5)
    
    postcode_label = ttk.Label(frame, text="Postcode:")
    postcode_label.pack(anchor='w')
    postcode_entry = ttk.Entry(frame)
    postcode_entry.pack(pady=5)
    
    password_label = ttk.Label(frame, text="Password:")
    password_label.pack(anchor='w')
    password_entry = ttk.Entry(frame, show="*")
    password_entry.pack(pady=5)
    
    confirm_password_label = ttk.Label(frame, text="Confirm Password:")
    confirm_password_label.pack(anchor='w')
    confirm_password_entry = ttk.Entry(frame, show="*")
    confirm_password_entry.pack(pady=5)
    
    # Register button
    register_button = ttk.Button(frame, text="Register", command=lambda: save_registration(first_name_entry.get(), last_name_entry.get(), email_entry.get(), phone_entry.get(), address_entry.get(), postcode_entry.get(), password_entry.get(), confirm_password_entry.get(), register_window))
    register_button.pack(pady=10)

def save_registration(first_name, last_name, email, phone, address, postcode, password, confirm_password, window):
    if not first_name or not last_name or not email or not address or not postcode or not password or not confirm_password:
        messagebox.showerror("Error", "All fields except phone are required.")
        return
    if not is_valid_email(email):
        messagebox.showerror("Invalid Email", "Please enter a valid email address.")
        return
    if phone and not is_valid_phone(phone):
        messagebox.showerror("Invalid Phone Number", "Please enter a valid phone number.")
        return
    if password != confirm_password:
        messagebox.showerror("Error", "Passwords do not match.")
        return
    # Save registration details (in a real application, you would save these details to a database)
    personal_details['first_name'] = first_name
    personal_details['last_name'] = last_name
    personal_details['email'] = email
    personal_details['phone'] = phone
    personal_details['address'] = address
    personal_details['postcode'] = postcode
    personal_details['password'] = password
    messagebox.showinfo("Success", "Registration successful!")
    window.destroy()

# Create the login window
login_window = tk.Tk()
login_window.title("Login")
login_window.configure(bg="#1e1e1e")

# Set the dimensions to resemble an iPhone screen
login_window.geometry("375x667")

# Create a frame for the login form
login_frame = ttk.Frame(login_window, padding=20)
login_frame.place(relx=0.1, rely=0.1, relwidth=0.8, relheight=0.8)

# Username label and entry
username_label = ttk.Label(login_frame, text="E-mail:")
username_label.pack(anchor='center')
username_entry = ttk.Entry(login_frame)
username_entry.pack(pady=5)

# Password label and entry
password_label = ttk.Label(login_frame, text="Password:")
password_label.pack(anchor='center')
password_entry = ttk.Entry(login_frame, show="*")
password_entry.pack(pady=5)

# Login button
login_button = ttk.Button(login_frame, text="Login", command=login)
login_button.pack(pady=10)

# Register button
register_button = ttk.Button(login_frame, text="Register", command=register)
register_button.pack(pady=5)

# Forgot Password button
forgot_password_button = ttk.Button(login_frame, text="Forgot Password", command=lambda: messagebox.showinfo("Forgot Password", "Please contact support to reset your password."))
forgot_password_button.pack(pady=5)

# Start the login GUI event loop
login_window.mainloop()
